In [15]:
import pandas as pd
import numpy as np
import time
from nba_api.stats.endpoints import playergamelog, leaguedashplayerstats

# WNBA seasons to pull — using calendar years not split years
WNBA_SEASONS = ['2021', '2022', '2023', '2024', '2025']

# WNBA team name mapping — all 13 current teams
team_name_map = {
    'ATL': 'Atlanta Dream',
    'CHI': 'Chicago Sky',
    'CON': 'Connecticut Sun',
    'DAL': 'Dallas Wings',
    'GSV': 'Golden State Valkyries',
    'IND': 'Indiana Fever',
    'LAS': 'Los Angeles Sparks',
    'LVA': 'Las Vegas Aces',
    'MIN': 'Minnesota Lynx',
    'NYL': 'New York Liberty',
    'PHO': 'Phoenix Mercury',   # 2021-2024
    'PHX': 'Phoenix Mercury',   # 2025 onwards
    'SEA': 'Seattle Storm',
    'WAS': 'Washington Mystics',
}

print("WNBA Pipeline initialized ✅")
print(f"Seasons to pull: {WNBA_SEASONS}")
print(f"Teams mapped:    {len(team_name_map)}")

WNBA Pipeline initialized ✅
Seasons to pull: ['2021', '2022', '2023', '2024', '2025']
Teams mapped:    14


In [19]:
# Pull players averaging 15+ minutes per game
# Lower threshold than NBA (20+) because WNBA rosters are smaller

all_qualified = []

for season in WNBA_SEASONS:
    stats = leaguedashplayerstats.LeagueDashPlayerStats(
        season=season,
        season_type_all_star='Regular Season',
        league_id_nullable='10'
    )
    time.sleep(1.5)

    df_s = stats.get_data_frames()[0]
    df_s['SEASON']  = season
    df_s['MIN_PG']  = df_s['MIN'] / df_s['GP']

    # Filter to 15+ min per game and 10+ games
    df_q = df_s[
        (df_s['MIN_PG'] >= 15) &
        (df_s['GP']     >= 10)
    ].copy()

    all_qualified.append(df_q)
    print(f"  {season}: {len(df_q)} qualified players")

df_qualified_all = pd.concat(all_qualified, ignore_index=True)

# Get unique players for the pull list
# Use most recent season's player list
df_qualified = all_qualified[-1].copy()
player_pull_list = df_qualified[['PLAYER_ID', 'PLAYER_NAME']].reset_index(drop=True)

print()
print(f"Total qualified player-seasons: {len(df_qualified_all):,}")
print(f"Players in latest season:       {len(player_pull_list)}")
print()
print("Sample qualified players:")
print(df_qualified[['PLAYER_NAME', 'MIN_PG', 'GP']]
      .sort_values('MIN_PG', ascending=False)
      .round(1)
      .head(10))

  2021: 101 qualified players
  2022: 95 qualified players
  2023: 90 qualified players
  2024: 96 qualified players
  2025: 108 qualified players

Total qualified player-seasons: 490
Players in latest season:       108

Sample qualified players:
          PLAYER_NAME  MIN_PG  GP
89        Kelsey Plum    35.1  43
149      Rhyne Howard    34.9  33
10       Allisha Gray    34.5  42
143    Paige Bueckers    33.3  36
18   Arike Ogunbowale    33.3  29
132  Napheesa Collier    32.3  33
165      Sonia Citron    32.1  44
55     Gabby Williams    31.6  44
16        Angel Reese    31.6  30
117     Marina Mabrey    31.5  35


In [21]:
def clean_player_log_wnba(df, player_name, season, first_season=False):
    """
    WNBA version of clean_player_log.
    Adapted for WNBA with:
    - Lower minute threshold (12 vs 15)
    - Relaxed min_periods for rolling averages (3 vs 5)
    - GAMES_PLAYED feature
    - IS_ROOKIE_SEASON flag
    """
    df['PLAYER_NAME'] = player_name
    df['SEASON']      = season
    df['GAME_DATE']   = pd.to_datetime(df['GAME_DATE'], format='mixed')
    df['HOME_AWAY']   = df['MATCHUP'].apply(
        lambda x: 'HOME' if 'vs.' in x else 'AWAY'
    )

    # Lower minute threshold for WNBA
    df = df[df['MIN'] >= 12].copy()

    # Sort oldest to newest
    df = df.sort_values('GAME_DATE').reset_index(drop=True)

    # Add games played counter — model uses this to gauge reliability
    df['GAMES_PLAYED'] = range(1, len(df) + 1)

    # Add rookie season flag
    df['IS_ROOKIE_SEASON'] = 1 if first_season else 0

    # Build rolling averages with relaxed min_periods
    # min_periods=3 means we keep rows after just 3 games
    target_stats       = ['PTS', 'REB', 'AST', 'BLK', 'STL', 'FG3M']
    feature_only_stats = ['MIN', 'TOV', 'FGA', 'FG3A']
    all_stats          = target_stats + feature_only_stats

    for stat in all_stats:
        df[f'{stat}_roll5']  = (
            df[stat].rolling(window=5, min_periods=3).mean().shift(1)
        )
        df[f'{stat}_roll10'] = (
            df[stat].rolling(window=10, min_periods=5).mean().shift(1)
        )

    # Drop only rows where we have less than 3 games of history
    df = df.dropna(subset=['PTS_roll5']).reset_index(drop=True)

    return df

print("✅ clean_player_log_wnba defined")

✅ clean_player_log_wnba defined


In [23]:
# Pull all WNBA game logs — all players all seasons
SEASONS = ['2021', '2022', '2023', '2024', '2025']
SLEEP   = 1.5

# Build full player list across all seasons
player_pull_list_full = df_qualified_all[
    ['PLAYER_ID', 'PLAYER_NAME']
].drop_duplicates().reset_index(drop=True)

print(f"Unique players to pull: {len(player_pull_list_full)}")
print()

# Track first season per player for rookie flag
player_first_season = {}

all_data = []
total    = len(player_pull_list_full) * len(SEASONS)
current  = 0

for _, player in player_pull_list_full.iterrows():
    player_name = player['PLAYER_NAME']
    found_first = False

    for season in SEASONS:
        current += 1
        try:
            log = playergamelog.PlayerGameLog(
                player_id=player['PLAYER_ID'],
                season=season,
                season_type_all_star='Regular Season',
                league_id_nullable='10'
            )
            time.sleep(SLEEP)

            df_raw = log.get_data_frames()[0]

            if len(df_raw) == 0:
                continue

            # First season with data = rookie season flag
            is_first = not found_first
            if is_first:
                found_first = True
                player_first_season[player_name] = season

            df_cleaned = clean_player_log_wnba(
                df_raw, player_name, season,
                first_season=is_first
            )

            if len(df_cleaned) == 0:
                continue

            all_data.append(df_cleaned)
            rookie_tag = '👶' if is_first else ''
            print(f"[{current}/{total}] ✅ {player_name} {season} "
                  f"— {len(df_cleaned)} rows {rookie_tag}")

        except Exception as e:
            print(f"[{current}/{total}] ❌ {player_name} {season} — {e}")
            time.sleep(SLEEP)
            continue

df_master_wnba = pd.concat(all_data, ignore_index=True)

print()
print(f"═══════════════════════════════════")
print(f"  Total rows:    {len(df_master_wnba):,}")
print(f"  Total players: {df_master_wnba['PLAYER_NAME'].nunique()}")
print(f"  Seasons:       {sorted(df_master_wnba['SEASON'].unique())}")
print(f"  Rookie seasons flagged: {len(player_first_season)}")
print(f"═══════════════════════════════════")
print()

# Show some notable rookies
notable = ['Caitlin Clark', 'Paige Bueckers', 'Angel Reese']
print("Notable rookie seasons:")
for player in notable:
    if player in player_first_season:
        print(f"  {player}: first season = {player_first_season[player]}")

Unique players to pull: 180

[1/900] ✅ A'ja Wilson 2021 — 29 rows 👶
[2/900] ✅ A'ja Wilson 2022 — 33 rows 
[3/900] ✅ A'ja Wilson 2023 — 37 rows 
[4/900] ✅ A'ja Wilson 2024 — 35 rows 
[5/900] ✅ A'ja Wilson 2025 — 37 rows 
[6/900] ✅ Aari McDonald 2021 — 17 rows 👶
[7/900] ✅ Aari McDonald 2022 — 33 rows 
[8/900] ✅ Aari McDonald 2023 — 21 rows 
[9/900] ✅ Aari McDonald 2024 — 17 rows 
[10/900] ✅ Aari McDonald 2025 — 16 rows 
[11/900] ✅ Aerial Powers 2021 — 10 rows 👶
[12/900] ✅ Aerial Powers 2022 — 32 rows 
[13/900] ✅ Aerial Powers 2023 — 3 rows 
[14/900] ✅ Aerial Powers 2024 — 13 rows 
[15/900] ✅ Aerial Powers 2025 — 5 rows 
[16/900] ✅ Allie Quigley 2021 — 23 rows 👶
[17/900] ✅ Allie Quigley 2022 — 31 rows 
[21/900] ✅ Allisha Gray 2021 — 21 rows 👶
[22/900] ✅ Allisha Gray 2022 — 30 rows 
[23/900] ✅ Allisha Gray 2023 — 35 rows 
[24/900] ✅ Allisha Gray 2024 — 37 rows 
[25/900] ✅ Allisha Gray 2025 — 39 rows 
[26/900] ✅ Amanda Zahui B 2021 — 25 rows 👶
[28/900] ✅ Amanda Zahui B 2023 — 4 rows 
[31/90

In [25]:
# Feature engineering — Days Rest
df_master_wnba = df_master_wnba.sort_values(
    ['PLAYER_NAME', 'GAME_DATE']
).reset_index(drop=True)

if 'DAYS_REST' in df_master_wnba.columns:
    df_master_wnba = df_master_wnba.drop(columns=['DAYS_REST'])

df_master_wnba['DAYS_REST'] = (
    df_master_wnba.groupby('PLAYER_NAME')['GAME_DATE']
    .diff()
    .dt.days
)
df_master_wnba['DAYS_REST'] = df_master_wnba['DAYS_REST'].fillna(2)
df_master_wnba['DAYS_REST'] = df_master_wnba['DAYS_REST'].clip(upper=7)

# Extract opponent from MATCHUP
df_master_wnba['OPPONENT'] = df_master_wnba['MATCHUP'].apply(
    lambda x: x.split()[-1]
)

# Map opponent to full team name
df_master_wnba['OPP_TEAM_NAME'] = df_master_wnba['OPPONENT'].map(team_name_map)

# Player team
df_master_wnba['PLAYER_TEAM'] = df_master_wnba['MATCHUP'].apply(
    lambda x: x.split(' vs.')[0].strip() if 'vs.' in x else x.split(' @')[0].strip()
)

print(f"NaNs in DAYS_REST:    {df_master_wnba['DAYS_REST'].isna().sum()}")
print(f"NaNs in OPPONENT:     {df_master_wnba['OPPONENT'].isna().sum()}")
print(f"NaNs in OPP_TEAM_NAME:{df_master_wnba['OPP_TEAM_NAME'].isna().sum()}")
print(f"Total columns:        {len(df_master_wnba.columns)}")
print()
print(df_master_wnba[['PLAYER_NAME', 'GAME_DATE', 'MATCHUP',
                       'OPPONENT', 'OPP_TEAM_NAME', 'DAYS_REST']].head(10))

NaNs in DAYS_REST:    0
NaNs in OPPONENT:     0
NaNs in OPP_TEAM_NAME:0
Total columns:        56

   PLAYER_NAME  GAME_DATE      MATCHUP OPPONENT       OPP_TEAM_NAME  DAYS_REST
0  A'ja Wilson 2021-05-23  LVA vs. CON      CON     Connecticut Sun        2.0
1  A'ja Wilson 2021-05-26    LVA @ PHO      PHO     Phoenix Mercury        3.0
2  A'ja Wilson 2021-05-28  LVA vs. IND      IND       Indiana Fever        2.0
3  A'ja Wilson 2021-05-30  LVA vs. IND      IND       Indiana Fever        2.0
4  A'ja Wilson 2021-06-01    LVA @ CON      CON     Connecticut Sun        2.0
5  A'ja Wilson 2021-06-03    LVA @ NYL      NYL    New York Liberty        2.0
6  A'ja Wilson 2021-06-05    LVA @ WAS      WAS  Washington Mystics        2.0
7  A'ja Wilson 2021-06-13  LVA vs. DAL      DAL        Dallas Wings        7.0
8  A'ja Wilson 2021-06-15  LVA vs. NYL      NYL    New York Liberty        2.0
9  A'ja Wilson 2021-06-17  LVA vs. NYL      NYL    New York Liberty        2.0


In [27]:
from nba_api.stats.endpoints import leaguedashteamstats
import time

# Pull team defensive stats for each WNBA season
opp_def_frames = []

for season in ['2021', '2022', '2023', '2024', '2025']:
    stats = leaguedashteamstats.LeagueDashTeamStats(
        season=season,
        season_type_all_star='Regular Season',
        measure_type_detailed_defense='Defense',
        league_id_nullable='10'
    )
    time.sleep(1.5)
    df_def = stats.get_data_frames()[0]
    df_def['SEASON'] = season
    opp_def_frames.append(df_def)
    print(f"  {season}: {len(df_def)} teams pulled")

df_opp_def = pd.concat(opp_def_frames, ignore_index=True)
df_opp_def = df_opp_def[['TEAM_NAME', 'SEASON', 'DEF_RATING']].copy()

# Pull pace stats
pace_frames = []

for season in ['2021', '2022', '2023', '2024', '2025']:
    stats = leaguedashteamstats.LeagueDashTeamStats(
        season=season,
        season_type_all_star='Regular Season',
        measure_type_detailed_defense='Advanced',
        league_id_nullable='10'
    )
    time.sleep(1.5)
    df_pace = stats.get_data_frames()[0]
    df_pace['SEASON'] = season
    pace_frames.append(df_pace)
    print(f"  {season}: pace pulled")

df_pace_all   = pd.concat(pace_frames, ignore_index=True)
df_pace_clean = df_pace_all[['TEAM_NAME', 'SEASON', 'PACE']].copy()

# Merge DEF_RATING
if 'DEF_RATING' in df_master_wnba.columns:
    df_master_wnba = df_master_wnba.drop(columns=['DEF_RATING'])

df_master_wnba = df_master_wnba.merge(
    df_opp_def[['TEAM_NAME', 'SEASON', 'DEF_RATING']],
    left_on=['OPP_TEAM_NAME', 'SEASON'],
    right_on=['TEAM_NAME', 'SEASON'],
    how='left'
).drop(columns=['TEAM_NAME'])

# Merge PACE
if 'PACE' in df_master_wnba.columns:
    df_master_wnba = df_master_wnba.drop(columns=['PACE'])

df_master_wnba = df_master_wnba.merge(
    df_pace_clean[['TEAM_NAME', 'SEASON', 'PACE']],
    left_on=['OPP_TEAM_NAME', 'SEASON'],
    right_on=['TEAM_NAME', 'SEASON'],
    how='left'
).drop(columns=['TEAM_NAME'])

print()
print(f"NaNs in DEF_RATING: {df_master_wnba['DEF_RATING'].isna().sum()}")
print(f"NaNs in PACE:       {df_master_wnba['PACE'].isna().sum()}")
print()
print(df_master_wnba[['PLAYER_NAME', 'OPPONENT', 'SEASON',
                       'DEF_RATING', 'PACE']].head(10))

  2021: 12 teams pulled
  2022: 12 teams pulled
  2023: 12 teams pulled
  2024: 12 teams pulled
  2025: 13 teams pulled
  2021: pace pulled
  2022: pace pulled
  2023: pace pulled
  2024: pace pulled
  2025: pace pulled

NaNs in DEF_RATING: 0
NaNs in PACE:       0

   PLAYER_NAME OPPONENT SEASON  DEF_RATING   PACE
0  A'ja Wilson      CON   2021        91.7  90.96
1  A'ja Wilson      PHO   2021       101.3  93.56
2  A'ja Wilson      IND   2021       107.8  94.78
3  A'ja Wilson      IND   2021       107.8  94.78
4  A'ja Wilson      CON   2021        91.7  90.96
5  A'ja Wilson      NYL   2021       104.3  97.81
6  A'ja Wilson      WAS   2021       104.4  95.79
7  A'ja Wilson      DAL   2021       103.0  94.47
8  A'ja Wilson      NYL   2021       104.3  97.81
9  A'ja Wilson      NYL   2021       104.3  97.81


In [29]:
# Pull opponent stats allowed per game for each WNBA season
opp_stats_frames = []

for season in ['2021', '2022', '2023', '2024', '2025']:
    stats = leaguedashteamstats.LeagueDashTeamStats(
        season=season,
        season_type_all_star='Regular Season',
        measure_type_detailed_defense='Base',
        per_mode_detailed='PerGame',
        league_id_nullable='10'
    )
    time.sleep(1.5)
    df_s = stats.get_data_frames()[0]
    df_s['SEASON'] = season
    opp_stats_frames.append(df_s)
    print(f"  {season}: {len(df_s)} teams pulled")

df_opp_stats = pd.concat(opp_stats_frames, ignore_index=True)

# Select and rename
df_opp_stats_clean = df_opp_stats[[
    'TEAM_NAME', 'SEASON',
    'PTS', 'REB', 'AST', 'BLK', 'STL', 'FG3M',
]].copy().rename(columns={
    'PTS':  'OPP_PTS_ALLOWED_PG',
    'REB':  'OPP_REB_ALLOWED_PG',
    'AST':  'OPP_AST_ALLOWED_PG',
    'BLK':  'OPP_BLK_PG',
    'STL':  'OPP_STL_PG',
    'FG3M': 'OPP_3PM_ALLOWED_PG',
})

# Protection
for col in ['OPP_PTS_ALLOWED_PG', 'OPP_REB_ALLOWED_PG', 'OPP_AST_ALLOWED_PG',
            'OPP_BLK_PG', 'OPP_STL_PG', 'OPP_3PM_ALLOWED_PG']:
    if col in df_master_wnba.columns:
        df_master_wnba = df_master_wnba.drop(columns=[col])

# Merge
df_master_wnba = df_master_wnba.merge(
    df_opp_stats_clean,
    left_on=['OPP_TEAM_NAME', 'SEASON'],
    right_on=['TEAM_NAME', 'SEASON'],
    how='left'
).drop(columns=['TEAM_NAME'])

print()
print(f"NaNs in OPP_PTS_ALLOWED_PG: {df_master_wnba['OPP_PTS_ALLOWED_PG'].isna().sum()}")
print(f"Total columns: {len(df_master_wnba.columns)}")
print()
print(df_master_wnba[['PLAYER_NAME', 'OPPONENT', 'SEASON',
                       'OPP_PTS_ALLOWED_PG', 'OPP_REB_ALLOWED_PG',
                       'OPP_3PM_ALLOWED_PG']].head(5))

  2021: 12 teams pulled
  2022: 12 teams pulled
  2023: 12 teams pulled
  2024: 12 teams pulled
  2025: 13 teams pulled

NaNs in OPP_PTS_ALLOWED_PG: 0
Total columns: 64

   PLAYER_NAME OPPONENT SEASON  OPP_PTS_ALLOWED_PG  OPP_REB_ALLOWED_PG  \
0  A'ja Wilson      CON   2021                79.7                36.6   
1  A'ja Wilson      PHO   2021                82.1                36.2   
2  A'ja Wilson      IND   2021                75.3                34.3   
3  A'ja Wilson      IND   2021                75.3                34.3   
4  A'ja Wilson      CON   2021                79.7                36.6   

   OPP_3PM_ALLOWED_PG  
0                 7.0  
1                 7.5  
2                 4.9  
3                 4.9  
4                 7.0  


In [31]:
# Pull WNBA usage stats for each season
from nba_api.stats.endpoints import leaguedashplayerstats

usage_frames = []

for season in ['2021', '2022', '2023', '2024', '2025']:
    stats = leaguedashplayerstats.LeagueDashPlayerStats(
        season=season,
        season_type_all_star='Regular Season',
        measure_type_detailed_defense='Advanced',
        per_mode_detailed='PerGame',
        league_id_nullable='10'
    )
    time.sleep(1.5)
    df_u = stats.get_data_frames()[0]
    df_u['SEASON'] = season
    usage_frames.append(df_u)
    print(f"  {season}: {len(df_u)} players pulled")

df_usage = pd.concat(usage_frames, ignore_index=True)

# Filter to GP >= 5 only — lower threshold for WNBA short season
df_usage_clean = df_usage[
    df_usage['GP'] >= 5
][['PLAYER_NAME', 'SEASON', 'USG_PCT']].copy()

# Protection
if 'USG_PCT' in df_master_wnba.columns:
    df_master_wnba = df_master_wnba.drop(columns=['USG_PCT'])

# Merge
df_master_wnba = df_master_wnba.merge(
    df_usage_clean[['PLAYER_NAME', 'SEASON', 'USG_PCT']],
    on=['PLAYER_NAME', 'SEASON'],
    how='left'
)

# Fill NaNs with league average
league_avg_usg = df_usage_clean['USG_PCT'].mean()
df_master_wnba['USG_PCT'] = df_master_wnba['USG_PCT'].fillna(league_avg_usg)

print()
print(f"NaNs in USG_PCT: {df_master_wnba['USG_PCT'].isna().sum()}")
print(f"League avg USG:  {league_avg_usg:.3f}")
print()

# Check star players
for player in ["A'ja Wilson", 'Caitlin Clark', 'Paige Bueckers']:
    row = df_master_wnba[
        df_master_wnba['PLAYER_NAME'] == player
    ].iloc[-1]
    print(f"  {player:<25} USG: {row['USG_PCT']:.3f}")

  2021: 155 players pulled
  2022: 166 players pulled
  2023: 156 players pulled
  2024: 157 players pulled
  2025: 182 players pulled

NaNs in USG_PCT: 0
League avg USG:  0.180

  A'ja Wilson               USG: 0.307
  Caitlin Clark             USG: 0.287
  Paige Bueckers            USG: 0.241


In [33]:
# WNBA position mapping
# G = Guard, F = Forward, C = Center

wnba_position_map = {
    "A'ja Wilson": 'C', 'Aaliyah Edwards': 'F', 'Aaliyah Nye': 'G',
    'Aari McDonald': 'G', 'Aerial Powers': 'F', 'Alanna Smith': 'F',
    'Alisha Gray': 'G', 'Alissa Pili': 'F', 'Aliyah Boston': 'F',
    'Allisha Gray': 'G', 'Allie Quigley': 'G', 'Alysha Clark': 'F',
    'Alyssa Thomas': 'F', 'Amanda Zahui B': 'C', 'Aneesah Morrow': 'F',
    'Angel Reese': 'C', 'Ariel Atkins': 'G', 'Arike Ogunbowale': 'G',
    'Astou Ndour-Fall': 'C', 'Azura Stevens': 'C', 'Aziaha James': 'G',
    'Betnijah Laney-Hamilton': 'G', 'Breanna Stewart': 'F',
    'Bria Hartley': 'G', 'Bria Holmes': 'G', 'Briann January': 'G',
    'Brianna Turner': 'F', 'Bridget Carleton': 'F', 'Brionna Jones': 'C',
    'Brittney Griner': 'C', 'Brittney Sykes': 'G',
    'Caitlin Clark': 'G', 'Cameron Brink': 'F', 'Candace Parker': 'F',
    'Candice Dupree': 'F', 'Carla Leite': 'G',
    'Cecilia Zandalasini': 'F', 'Chelsea Gray': 'G',
    'Chennedy Carter': 'G', 'Cheyenne Parker-Tyus': 'F',
    'Chiney Ogwumike': 'C', 'Courtney Vandersloot': 'G',
    'Courtney Williams': 'G', 'Crystal Bradford': 'F',
    'Crystal Dangerfield': 'G', 'Dana Evans': 'G',
    'Damiris Dantas': 'F', 'Danielle Robinson': 'G',
    'DeWanna Bonner': 'F', 'Dearica Hamby': 'F',
    'Destanni Henderson': 'G', 'Diamond DeShields': 'G',
    'Diamond Miller': 'G', 'Diana Taurasi': 'G',
    'DiJonai Carrington': 'G', 'Dorka Juhasz': 'F',
    'Elena Delle Donne': 'F', 'Elizabeth Williams': 'C',
    'Emily Engstler': 'F', 'Emma Meesseman': 'F',
    'Erica McCall': 'F', 'Erica Wheeler': 'G',
    'Ezi Magbegor': 'C', 'Gabby Williams': 'F',
    'Grace Berger': 'G', 'Haley Jones': 'F',
    'Han Xu': 'C', 'Iliana Rupert': 'C',
    'Isabelle Harrison': 'F', 'Ivana Dojkić': 'G',
    'JJ Quinerly': 'G', 'Jackie Young': 'G',
    'Jacy Sheldon': 'G', 'Jade Melbourne': 'G',
    'Janelle Salaun': 'F', 'Jantel Lavender': 'C',
    'Jasmine Thomas': 'G', 'Jazmine Jones': 'G',
    'Jewell Loyd': 'G', 'Jessica Breland': 'F',
    'Jessica Shepard': 'C', 'Jonquel Jones': 'C',
    'Jordan Horston': 'G', 'Jordin Canada': 'G',
    'Julie Allemand': 'G', 'Julie Vanloo': 'G',
    'Kahleah Copper': 'G', 'Kaila Charles': 'F',
    'Kalani Brown': 'C', 'Kamilla Cardoso': 'C',
    'Karlie Samuelson': 'G', 'Kate Martin': 'G',
    'Kathryn Westbeld': 'F', 'Katie Lou Samuelson': 'G',
    'Kayla McBride': 'G', 'Kayla Thornton': 'F',
    'Kelsey Mitchell': 'G', 'Kelsey Plum': 'G',
    'Kennedy Burke': 'G', 'Kia Nurse': 'G',
    'Kia Vaughn': 'C', 'Kiah Stokes': 'C',
    'Kiki Iriafen': 'F', 'Kitija Laksa': 'G',
    'Kristi Toliver': 'G', 'Kristy Wallace': 'G',
    'Kylee Shook': 'F', 'LaToya Sanders': 'F',
    'Layshia Clarendon': 'G', 'Leila Lacan': 'G',
    'Leione Fiebich': 'G', 'Leonie Fiebich': 'G',
    'Leilani Mitchell': 'G', 'Lexie Brown': 'G',
    'Lexie Hull': 'G', 'Li Meng': 'G',
    'Li Yueru': 'C', 'Lindsay Allen': 'G',
    'Liz Cambage': 'C', 'Luisa Geiselsoder': 'F',
    'Maddy Siegrist': 'F', 'Marine Johannes': 'G',
    'Marina Mabrey': 'G', 'Maya Caldwell': 'G',
    'Megan Gustafson': 'C', 'Mercedes Russell': 'C',
    'Michaela Onyenwere': 'F', 'Monique Akoa Makani': 'F',
    'Monique Billings': 'F', 'Moriah Jefferson': 'G',
    'Myisha Hines-Allen': 'F', 'NaLyssa Smith': 'F',
    'Napheesa Collier': 'F', 'Natalie Achonwa': 'F',
    'Natasha Cloud': 'G', 'Natasha Howard': 'F',
    'Natasha Mack': 'C', 'Natisha Hiedeman': 'G',
    'Naz Hillmon': 'C', 'Nia Coffey': 'F',
    'Nneka Ogwumike': 'F', 'Nyara Sabally': 'F',
    'Odyssey Sims': 'G', 'Olivia Nelson-Ododa': 'C',
    'Paige Bueckers': 'G', 'Queen Egbo': 'C',
    'Rachel Banham': 'G', 'Rae Burrell': 'G',
    'Rebecca Allen': 'F', 'Rebekah Gardner': 'G',
    'Rhyne Howard': 'G', 'Rickea Jackson': 'F',
    'Riquna Williams': 'G', 'Ruthy Hebard': 'F',
    'Sabrina Ionescu': 'G', 'Sami Whitcomb': 'G',
    'Saniya Rivers': 'G', 'Satou Sabally': 'F',
    'Sevgi Uzun': 'G', 'Shakira Austin': 'C',
    'Shatori Walker-Kimbrough': 'G', 'Shavonte Zellous': 'G',
    'Shey Peddy': 'G', 'Skylar Diggins': 'G',
    'Sonia Citron': 'G', 'Sophie Cunningham': 'G',
    'Stefanie Dolson': 'C', 'Stephanie Talbot': 'F',
    'Sue Bird': 'G', 'Sug Sutton': 'G',
    'Sydney Wiese': 'G', 'Sylvia Fowles': 'C',
    'Te-Hina Paopao': 'G', 'Te\'a Cooper': 'G',
    'Teaira McCowan': 'C', 'Temi Fagbenle': 'C',
    'Theresa Plaisance': 'C', 'Tianna Hawkins': 'F',
    'Tiffany Hayes': 'G', 'Tiffany Mitchell': 'G',
    'Tina Charles': 'C', 'Tyasha Harris': 'G',
    'Veronica Burton': 'G', 'Victoria Vivians': 'G',
    'JJ Quinerly': 'G', 'Janelle Salaun': 'F',
    'Carla Leite': 'G', 'Iliana Rupert': 'C',
    'Cecilia Zandalasini': 'F', 'Kaila Charles': 'F',
    'Leila Lacan': 'G', 'Luisa Geiselsoder': 'F',
    'Monique Akoa Makani': 'F', 'Kiki Iriafen': 'F',
    'Kitija Laksa': 'G', 'Aaliyah Nye': 'G',
    'Aneesah Morrow': 'F', 'Aziaha James': 'G',
    'Saniya Rivers': 'G', 'Sonia Citron': 'G',
    'Te-Hina Paopao': 'G', 'Kathryn Westbeld': 'F',
    'Janelle Salaun': 'F', 'Kate Martin': 'G',
}

df_master_wnba['POSITION'] = df_master_wnba['PLAYER_NAME'].map(wnba_position_map)

mapped   = df_master_wnba['POSITION'].notna().sum()
total    = len(df_master_wnba)
unmapped = df_master_wnba[
    df_master_wnba['POSITION'].isna()
]['PLAYER_NAME'].unique()

print(f"Players mapped: {mapped:,} / {total:,} ({mapped/total*100:.1f}%)")
print()
print(f"Unmapped players ({len(unmapped)}):")
print(unmapped if len(unmapped) > 0 else "None ✅")

Players mapped: 14,597 / 14,597 (100.0%)

Unmapped players (0):
None ✅


In [35]:
# Add opponent vs position stats
df_pos_stats = df_master_wnba[[
    'OPPONENT', 'SEASON', 'POSITION',
    'PTS', 'REB', 'AST', 'BLK', 'STL', 'FG3M'
]].copy()

pos_def = df_pos_stats.groupby(['OPPONENT', 'SEASON', 'POSITION'])[
    ['PTS', 'REB', 'AST', 'BLK', 'STL', 'FG3M']
].mean().round(2).reset_index()

pos_def = pos_def.rename(columns={
    'PTS':  'OPP_PTS_VS_POS', 'REB':  'OPP_REB_VS_POS',
    'AST':  'OPP_AST_VS_POS', 'BLK':  'OPP_BLK_VS_POS',
    'STL':  'OPP_STL_VS_POS', 'FG3M': 'OPP_3PM_VS_POS',
})

for col in ['OPP_PTS_VS_POS', 'OPP_REB_VS_POS', 'OPP_AST_VS_POS',
            'OPP_BLK_VS_POS', 'OPP_STL_VS_POS', 'OPP_3PM_VS_POS']:
    if col in df_master_wnba.columns:
        df_master_wnba = df_master_wnba.drop(columns=[col])

df_master_wnba = df_master_wnba.merge(
    pos_def, on=['OPPONENT', 'SEASON', 'POSITION'], how='left'
)

# Rolling team-vs-position features
merge_cols = {
    'ALLOWED_PTS_roll5':  'OPP_PTS_VS_POS_roll5',
    'ALLOWED_REB_roll5':  'OPP_REB_VS_POS_roll5',
    'ALLOWED_AST_roll5':  'OPP_AST_VS_POS_roll5',
    'ALLOWED_BLK_roll5':  'OPP_BLK_VS_POS_roll5',
    'ALLOWED_STL_roll5':  'OPP_STL_VS_POS_roll5',
    'ALLOWED_FG3M_roll5': 'OPP_3PM_VS_POS_roll5',
}

stats_to_roll = ['PTS', 'REB', 'AST', 'BLK', 'STL', 'FG3M']

team_vs_pos_daily = df_master_wnba.groupby(
    ['OPPONENT', 'GAME_DATE', 'POSITION']
)[stats_to_roll].sum().reset_index()

team_vs_pos_daily = team_vs_pos_daily.rename(columns={
    'PTS': 'ALLOWED_PTS', 'REB': 'ALLOWED_REB', 'AST': 'ALLOWED_AST',
    'BLK': 'ALLOWED_BLK', 'STL': 'ALLOWED_STL', 'FG3M': 'ALLOWED_FG3M',
})

team_vs_pos_daily = team_vs_pos_daily.sort_values(
    ['OPPONENT', 'POSITION', 'GAME_DATE']
).reset_index(drop=True)

allowed_stats = ['ALLOWED_PTS', 'ALLOWED_REB', 'ALLOWED_AST',
                 'ALLOWED_BLK', 'ALLOWED_STL', 'ALLOWED_FG3M']

for stat in allowed_stats:
    team_vs_pos_daily[f'{stat}_roll5'] = (
        team_vs_pos_daily.groupby(['OPPONENT', 'POSITION'])[stat]
        .transform(lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))
    )

# Convert to per-player averages
position_counts = df_master_wnba.groupby(
    ['OPPONENT', 'GAME_DATE', 'POSITION']
).size().reset_index(name='player_count')

position_counts = position_counts.sort_values(
    ['OPPONENT', 'POSITION', 'GAME_DATE']
)
position_counts['player_count_roll5'] = (
    position_counts.groupby(['OPPONENT', 'POSITION'])['player_count']
    .transform(lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))
)

team_vs_pos_daily = team_vs_pos_daily.merge(
    position_counts[['OPPONENT', 'GAME_DATE', 'POSITION', 'player_count_roll5']],
    on=['OPPONENT', 'GAME_DATE', 'POSITION'],
    how='left'
)

for col in merge_cols.keys():
    team_vs_pos_daily[col] = (
        team_vs_pos_daily[col] / team_vs_pos_daily['player_count_roll5']
    )

team_vs_pos_merge = team_vs_pos_daily[
    ['OPPONENT', 'GAME_DATE', 'POSITION'] + list(merge_cols.keys())
].rename(columns=merge_cols)

for col in merge_cols.values():
    if col in df_master_wnba.columns:
        df_master_wnba = df_master_wnba.drop(columns=[col])

df_master_wnba = df_master_wnba.merge(
    team_vs_pos_merge,
    on=['OPPONENT', 'GAME_DATE', 'POSITION'],
    how='left'
)

# Fill NaNs with static averages
roll_cols   = list(merge_cols.values())
static_cols = ['OPP_PTS_VS_POS', 'OPP_REB_VS_POS', 'OPP_AST_VS_POS',
               'OPP_BLK_VS_POS', 'OPP_STL_VS_POS', 'OPP_3PM_VS_POS']

for roll_col, static_col in zip(roll_cols, static_cols):
    df_master_wnba[roll_col] = df_master_wnba[roll_col].fillna(
        df_master_wnba[static_col]
    )

# Add position-normalized rolling features
pos_roll_cols = [
    'OPP_PTS_VS_POS_roll5', 'OPP_REB_VS_POS_roll5',
    'OPP_AST_VS_POS_roll5', 'OPP_BLK_VS_POS_roll5',
    'OPP_STL_VS_POS_roll5', 'OPP_3PM_VS_POS_roll5'
]

for col in pos_roll_cols:
    norm_col = f'{col}_norm'
    df_master_wnba[norm_col] = df_master_wnba.groupby('POSITION')[col].transform(
        lambda x: (x - x.mean()) / (x.std() + 1e-8)
    )

# Rolling usage rate
df_master_wnba = df_master_wnba.sort_values(
    ['PLAYER_NAME', 'GAME_DATE']
).reset_index(drop=True)

if 'USG_PCT_roll5' in df_master_wnba.columns:
    df_master_wnba = df_master_wnba.drop(columns=['USG_PCT_roll5'])

df_master_wnba['USG_PCT_roll5'] = (
    df_master_wnba.groupby('PLAYER_NAME')['USG_PCT']
    .transform(lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))
)
df_master_wnba['USG_PCT_roll5'] = df_master_wnba['USG_PCT_roll5'].fillna(
    df_master_wnba['USG_PCT']
)

# Team usage context
df_master_wnba['PLAYER_TEAM'] = df_master_wnba['MATCHUP'].apply(
    lambda x: x.split(' vs.')[0].strip() if 'vs.' in x else x.split(' @')[0].strip()
)

team_avg_usg = df_master_wnba.groupby(
    ['PLAYER_TEAM', 'SEASON']
)['USG_PCT'].mean().reset_index()
team_avg_usg = team_avg_usg.rename(columns={'USG_PCT': 'TEAM_AVG_USG'})

if 'TEAM_AVG_USG' in df_master_wnba.columns:
    df_master_wnba = df_master_wnba.drop(columns=['TEAM_AVG_USG'])

df_master_wnba = df_master_wnba.merge(
    team_avg_usg, on=['PLAYER_TEAM', 'SEASON'], how='left'
)

if 'RELATIVE_USG' in df_master_wnba.columns:
    df_master_wnba = df_master_wnba.drop(columns=['RELATIVE_USG'])

df_master_wnba['RELATIVE_USG'] = (
    df_master_wnba['USG_PCT'] / df_master_wnba['TEAM_AVG_USG']
)

player_season_usg = df_master_wnba.groupby(
    ['PLAYER_NAME', 'PLAYER_TEAM', 'SEASON']
)['USG_PCT'].first().reset_index()

player_season_usg['USG_RANK'] = player_season_usg.groupby(
    ['PLAYER_TEAM', 'SEASON']
)['USG_PCT'].rank(ascending=False, method='min')

if 'USG_RANK' in df_master_wnba.columns:
    df_master_wnba = df_master_wnba.drop(columns=['USG_RANK'])

df_master_wnba = df_master_wnba.merge(
    player_season_usg[['PLAYER_NAME', 'PLAYER_TEAM', 'SEASON', 'USG_RANK']],
    on=['PLAYER_NAME', 'PLAYER_TEAM', 'SEASON'],
    how='left'
)

# Volatility features
df_master_wnba = df_master_wnba.sort_values(
    ['PLAYER_NAME', 'GAME_DATE']
).reset_index(drop=True)

for col in ['PTS_std_roll10', 'REB_std_roll10',
            'PTS_cv_roll10', 'REB_cv_roll10']:
    if col in df_master_wnba.columns:
        df_master_wnba = df_master_wnba.drop(columns=[col])

df_master_wnba['PTS_std_roll10'] = (
    df_master_wnba.groupby('PLAYER_NAME')['PTS']
    .transform(lambda x: x.rolling(window=10, min_periods=3).std().shift(1))
)
df_master_wnba['REB_std_roll10'] = (
    df_master_wnba.groupby('PLAYER_NAME')['REB']
    .transform(lambda x: x.rolling(window=10, min_periods=3).std().shift(1))
)

df_master_wnba['PTS_cv_roll10'] = (
    df_master_wnba['PTS_std_roll10'] /
    df_master_wnba['PTS_roll10'].replace(0, np.nan)
)
df_master_wnba['REB_cv_roll10'] = (
    df_master_wnba['REB_std_roll10'] /
    df_master_wnba['REB_roll10'].replace(0, np.nan)
)

for col in ['PTS_std_roll10', 'REB_std_roll10',
            'PTS_cv_roll10', 'REB_cv_roll10']:
    df_master_wnba[col] = df_master_wnba[col].fillna(
        df_master_wnba[col].mean()
    )

# Final verify
print(f"Total rows:    {len(df_master_wnba):,}")
print(f"Total columns: {len(df_master_wnba.columns)}")
print()

check_cols = [
    'OPP_PTS_VS_POS', 'OPP_PTS_VS_POS_roll5',
    'OPP_PTS_VS_POS_roll5_norm', 'USG_PCT_roll5',
    'RELATIVE_USG', 'USG_RANK',
    'PTS_std_roll10', 'PTS_cv_roll10',
    'IS_ROOKIE_SEASON', 'GAMES_PLAYED'
]

print("Feature check:")
for col in check_cols:
    nans   = df_master_wnba[col].isna().sum()
    status = "✅" if nans == 0 else f"❌ {nans} NaNs"
    print(f"  {col:<35} {status}")

Total rows:    14,597
Total columns: 92

Feature check:
  OPP_PTS_VS_POS                      ✅
  OPP_PTS_VS_POS_roll5                ✅
  OPP_PTS_VS_POS_roll5_norm           ✅
  USG_PCT_roll5                       ✅
  RELATIVE_USG                        ✅
  USG_RANK                            ✅
  PTS_std_roll10                      ✅
  PTS_cv_roll10                       ✅
  IS_ROOKIE_SEASON                    ✅
  GAMES_PLAYED                        ✅


In [37]:
# Save WNBA dataset
df_master_wnba.to_csv('wnba_master_dataset.csv', index=False)

df_verify = pd.read_csv('wnba_master_dataset.csv', nrows=1)

print(f"✅ Saved successfully")
print(f"Rows:    {len(df_master_wnba):,}")
print(f"Columns: {len(df_verify.columns)}")
print()

# Final summary
print("Dataset summary:")
print(f"  Players:         {df_master_wnba['PLAYER_NAME'].nunique()}")
print(f"  Seasons:         {sorted(df_master_wnba['SEASON'].unique())}")
print(f"  Rookie rows:     {df_master_wnba['IS_ROOKIE_SEASON'].sum():,}")
print(f"  Regular rows:    {(df_master_wnba['IS_ROOKIE_SEASON']==0).sum():,}")
print()

# Spot check key players
print("Spot check:")
for player in ["A'ja Wilson", 'Caitlin Clark', 'Paige Bueckers', 'Sabrina Ionescu']:
    rows = df_master_wnba[df_master_wnba['PLAYER_NAME'] == player]
    if len(rows) == 0:
        continue
    latest = rows.iloc[-1]
    print(f"  {player:<25} Games: {len(rows):>3}  "
          f"USG: {latest['USG_PCT']:.3f}  "
          f"RelUSG: {latest['RELATIVE_USG']:.2f}  "
          f"Rank: {int(latest['USG_RANK'])}  "
          f"Rookie: {int(latest['IS_ROOKIE_SEASON'])}")

✅ Saved successfully
Rows:    14,597
Columns: 92

Dataset summary:
  Players:         180
  Seasons:         ['2021', '2022', '2023', '2024', '2025']
  Rookie rows:     3,856
  Regular rows:    10,741

Spot check:
  A'ja Wilson               Games: 171  USG: 0.307  RelUSG: 1.62  Rank: 1  Rookie: 0
  Caitlin Clark             Games:  47  USG: 0.287  RelUSG: 1.49  Rank: 1  Rookie: 0
  Paige Bueckers            Games:  33  USG: 0.241  RelUSG: 1.27  Rank: 1  Rookie: 1
  Sabrina Ionescu           Games: 163  USG: 0.273  RelUSG: 1.45  Rank: 1  Rookie: 0


In [11]:
# Step 4 — Pull complete 2026 WNBA season player game logs
from nba_api.stats.endpoints import playergamelog, leaguedashplayerstats

print("Pulling 2026 WNBA season data...")
print()

season_stats_2026 = leaguedashplayerstats.LeagueDashPlayerStats(
    season='2026',
    season_type_all_star='Regular Season',
    league_id_nullable='10'
)
time.sleep(1.5)
df_2026 = season_stats_2026.get_data_frames()[0]
df_2026['MIN_PG'] = df_2026['MIN'] / df_2026['GP']

df_qualified_2026 = df_2026[
    (df_2026['MIN_PG'] >= 15) &
    (df_2026['GP']     >= 3)
].copy()

player_pull_2026 = df_qualified_2026[
    ['PLAYER_ID', 'PLAYER_NAME']
].reset_index(drop=True)

print(f"Qualified players in 2026: {len(player_pull_2026)}")
print()

all_data_2026 = []
total         = len(player_pull_2026)
current       = 0

existing_players = set(df_master_wnba['PLAYER_NAME'].unique())

for _, player in player_pull_2026.iterrows():
    current += 1
    try:
        log = playergamelog.PlayerGameLog(
            player_id=player['PLAYER_ID'],
            season='2026',
            season_type_all_star='Regular Season',
            league_id_nullable='10'
        )
        time.sleep(1.5)
        df_raw = log.get_data_frames()[0]

        if len(df_raw) == 0:
            continue

        is_new = player['PLAYER_NAME'] not in existing_players

        df_cleaned = clean_player_log_wnba(
            df_raw,
            player['PLAYER_NAME'],
            2026,   # int to match SEASON dtype
            first_season=is_new
        )

        if len(df_cleaned) == 0:
            continue

        all_data_2026.append(df_cleaned)
        new_tag = '🆕' if is_new else ''
        print(f"[{current}/{total}] ✅ {player['PLAYER_NAME']} 2026 "
              f"— {len(df_cleaned)} rows {new_tag}")

    except Exception as e:
        print(f"[{current}/{total}] ❌ {player['PLAYER_NAME']} — {e}")
        time.sleep(1.5)
        continue

if len(all_data_2026) == 0:
    print("No 2026 data pulled")
else:
    df_2026_clean = pd.concat(all_data_2026, ignore_index=True)
    print()
    print(f"═══════════════════════════════════")
    print(f"  2026 rows pulled:  {len(df_2026_clean):,}")
    print(f"  Players:           {df_2026_clean['PLAYER_NAME'].nunique()}")
    print(f"  New players:       {df_2026_clean[df_2026_clean['IS_ROOKIE_SEASON']==1]['PLAYER_NAME'].nunique()}")
    print(f"═══════════════════════════════════")

Pulling 2026 WNBA season data...

Qualified players in 2026: 118

[1/118] ✅ A'ja Wilson 2026 — 14 rows 
[2/118] ✅ Aaliyah Edwards 2026 — 8 rows 
[3/118] ✅ Alanna Smith 2026 — 6 rows 
[4/118] ✅ Alicia Florez 2026 — 3 rows 🆕
[5/118] ✅ Aliyah Boston 2026 — 13 rows 
[6/118] ✅ Allisha Gray 2026 — 13 rows 
[7/118] ✅ Alyssa Thomas 2026 — 14 rows 
[8/118] ✅ Aneesah Morrow 2026 — 11 rows 
[9/118] ✅ Angel Reese 2026 — 13 rows 
[10/118] ✅ Angela Dugalic 2026 — 9 rows 🆕
[11/118] ✅ Ariel Atkins 2026 — 11 rows 
[12/118] ✅ Arike Ogunbowale 2026 — 14 rows 
[13/118] ✅ Awa Fam 2026 — 9 rows 🆕
[14/118] ✅ Awak Kuier 2026 — 8 rows 🆕
[15/118] ✅ Azura Stevens 2026 — 8 rows 
[16/118] ✅ Azzi Fudd 2026 — 13 rows 🆕
[17/118] ✅ Betnijah Laney-Hamilton 2026 — 8 rows 
[18/118] ✅ Breanna Stewart 2026 — 15 rows 
[19/118] ✅ Bridget Carleton 2026 — 13 rows 
[20/118] ✅ Brittney Griner 2026 — 7 rows 
[21/118] ✅ Brittney Sykes 2026 — 12 rows 
[22/118] ✅ Caitlin Clark 2026 — 13 rows 
[23/118] ✅ Cameron Brink 2026 — 9 rows 


In [17]:
# Fix Portland Fire abbreviation
team_name_map['PDX'] = 'Portland Fire'
team_name_map.pop('POR', None)  # remove the incorrect one

print("Updated team_name_map check:")
print(team_name_map)

Updated team_name_map check:
{'ATL': 'Atlanta Dream', 'CHI': 'Chicago Sky', 'CON': 'Connecticut Sun', 'DAL': 'Dallas Wings', 'GSV': 'Golden State Valkyries', 'IND': 'Indiana Fever', 'LAS': 'Los Angeles Sparks', 'LVA': 'Las Vegas Aces', 'MIN': 'Minnesota Lynx', 'NYL': 'New York Liberty', 'PHO': 'Phoenix Mercury', 'PHX': 'Phoenix Mercury', 'SEA': 'Seattle Storm', 'WAS': 'Washington Mystics', 'TOR': 'Toronto Tempo', 'PDX': 'Portland Fire'}


In [19]:
# Rebuild wnba_position_map from existing historical data
wnba_position_map = (
    df_master_wnba[['PLAYER_NAME', 'POSITION']]
    .drop_duplicates(subset='PLAYER_NAME')
    .set_index('PLAYER_NAME')['POSITION']
    .to_dict()
)

print(f"Loaded position map for {len(wnba_position_map)} existing players")

Loaded position map for 180 existing players


In [21]:
# Step 5 — Feature engineering for 2026 rows using REAL 2026 team stats

# Days rest
df_2026_clean = df_2026_clean.sort_values(
    ['PLAYER_NAME', 'GAME_DATE']
).reset_index(drop=True)

df_2026_clean['DAYS_REST'] = (
    df_2026_clean.groupby('PLAYER_NAME')['GAME_DATE']
    .diff().dt.days.fillna(2).clip(upper=7)
)

# Opponent and team extraction
df_2026_clean['OPPONENT'] = df_2026_clean['MATCHUP'].apply(lambda x: x.split()[-1])
df_2026_clean['PLAYER_TEAM'] = df_2026_clean['MATCHUP'].apply(
    lambda x: x.split(' vs.')[0].strip() if 'vs.' in x else x.split(' @')[0].strip()
)
df_2026_clean['OPP_TEAM_NAME'] = df_2026_clean['OPPONENT'].map(team_name_map)

# Check unmapped opponents (catches new abbreviations like POR/TOR if missing)
unmapped_opp = df_2026_clean[df_2026_clean['OPP_TEAM_NAME'].isna()]['OPPONENT'].unique()
print(f"Unmapped opponent abbreviations: "
      f"{unmapped_opp if len(unmapped_opp) > 0 else 'None ✅'}")

# Position mapping — reuse existing map, extend for new rookies
new_position_map = {
    'Alicia Florez':          'G',
    'Angela Dugalic':         'F',
    'Awa Fam':                'C',
    'Awak Kuier':             'F',
    'Azzi Fudd':              'G',
    'Charlisse Leger-Walker': 'G',
    'Cotie McMahon':          'F',
    'Dominique Malonga':      'C',
    'Flau\'jae Johnson':      'G',
    'Gabriela Jaquez':        'F',
    'Georgia Amoore':         'G',
    'Hailey Van Lith':        'G',
    'Jovana Nogic':           'F',
    'Kiki Rice':              'G',
    'Laura Juskaite':         'C',
    'Lauren Betts':           'C',
    'Maria Conde':            'G',
    'Noemie Brochant':        'G',
    'Nyadiew Puoch':          'C',
    'Olivia Miles':           'G',
    'Pauline Astier':         'F',
    'Sarah Ashlee Barker':    'G',
    'Sydney Taylor':          'G',
    'Zia Cooke':              'G',
}

full_position_map = {**wnba_position_map, **new_position_map}
df_2026_clean['POSITION'] = df_2026_clean['PLAYER_NAME'].map(full_position_map)

unmapped_pos = df_2026_clean[df_2026_clean['POSITION'].isna()]['PLAYER_NAME'].unique()
print(f"Unmapped player positions: "
      f"{unmapped_pos if len(unmapped_pos) > 0 else 'None ✅'}")

# Merge REAL 2026 team stats (no longer using 2025 proxy)
df_2026_clean = df_2026_clean.merge(
    df_def_2026[['TEAM_NAME', 'DEF_RATING']],
    left_on='OPP_TEAM_NAME', right_on='TEAM_NAME', how='left'
).drop(columns=['TEAM_NAME'])

df_2026_clean = df_2026_clean.merge(
    df_pace_2026[['TEAM_NAME', 'PACE']],
    left_on='OPP_TEAM_NAME', right_on='TEAM_NAME', how='left'
).drop(columns=['TEAM_NAME'])

df_2026_clean = df_2026_clean.merge(
    df_opp_stats_2026[['TEAM_NAME', 'OPP_PTS_ALLOWED_PG', 'OPP_REB_ALLOWED_PG',
                        'OPP_AST_ALLOWED_PG', 'OPP_BLK_PG', 'OPP_STL_PG',
                        'OPP_3PM_ALLOWED_PG']],
    left_on='OPP_TEAM_NAME', right_on='TEAM_NAME', how='left'
).drop(columns=['TEAM_NAME'])

df_2026_clean['SEASON'] = 2026

print()
print(f"NaNs in DEF_RATING:         {df_2026_clean['DEF_RATING'].isna().sum()}")
print(f"NaNs in PACE:               {df_2026_clean['PACE'].isna().sum()}")
print(f"NaNs in OPP_PTS_ALLOWED_PG: {df_2026_clean['OPP_PTS_ALLOWED_PG'].isna().sum()}")
print(f"Total rows: {len(df_2026_clean):,}")

Unmapped opponent abbreviations: None ✅
Unmapped player positions: None ✅

NaNs in DEF_RATING:         0
NaNs in PACE:               0
NaNs in OPP_PTS_ALLOWED_PG: 0
Total rows: 1,285


In [29]:
import numpy as np

# Step 5 continued — remaining features for 2026 rows (idempotent version)

# Opponent vs position stats — use 2026 data directly
for col in ['OPP_PTS_VS_POS', 'OPP_REB_VS_POS', 'OPP_AST_VS_POS',
            'OPP_BLK_VS_POS', 'OPP_STL_VS_POS', 'OPP_3PM_VS_POS']:
    if col in df_2026_clean.columns:
        df_2026_clean = df_2026_clean.drop(columns=[col])

pos_def_2026 = df_2026_clean.groupby(
    ['OPPONENT', 'POSITION']
)[['PTS', 'REB', 'AST', 'BLK', 'STL', 'FG3M']].mean().round(2).reset_index()

pos_def_2026 = pos_def_2026.rename(columns={
    'PTS':  'OPP_PTS_VS_POS', 'REB':  'OPP_REB_VS_POS',
    'AST':  'OPP_AST_VS_POS', 'BLK':  'OPP_BLK_VS_POS',
    'STL':  'OPP_STL_VS_POS', 'FG3M': 'OPP_3PM_VS_POS',
})

df_2026_clean = df_2026_clean.merge(
    pos_def_2026, on=['OPPONENT', 'POSITION'], how='left'
)

for col in ['OPP_PTS_VS_POS', 'OPP_REB_VS_POS', 'OPP_AST_VS_POS',
            'OPP_BLK_VS_POS', 'OPP_STL_VS_POS', 'OPP_3PM_VS_POS']:
    df_2026_clean[col] = df_2026_clean[col].fillna(df_2026_clean[col].mean())
    roll_col = f'{col}_roll5'
    if roll_col in df_2026_clean.columns:
        df_2026_clean = df_2026_clean.drop(columns=[roll_col])
    df_2026_clean[roll_col] = df_2026_clean[col]

# Usage rate — pull real 2026 usage stats
from nba_api.stats.endpoints import leaguedashplayerstats

if 'USG_PCT' in df_2026_clean.columns:
    df_2026_clean = df_2026_clean.drop(columns=['USG_PCT'])

usage_2026 = leaguedashplayerstats.LeagueDashPlayerStats(
    season='2026',
    season_type_all_star='Regular Season',
    measure_type_detailed_defense='Advanced',
    per_mode_detailed='PerGame',
    league_id_nullable='10'
)
time.sleep(1.5)
df_usg_2026 = usage_2026.get_data_frames()[0]
df_usg_2026['SEASON'] = 2026

# Drop any duplicate PLAYER_NAME/SEASON rows (traded players appear twice — keep first)
df_usg_2026_clean = df_usg_2026[
    df_usg_2026['GP'] >= 3
][['PLAYER_NAME', 'SEASON', 'USG_PCT']].drop_duplicates(
    subset=['PLAYER_NAME', 'SEASON']
).copy()

df_2026_clean = df_2026_clean.merge(
    df_usg_2026_clean, on=['PLAYER_NAME', 'SEASON'], how='left'
)

league_avg_usg_2026 = df_usg_2026_clean['USG_PCT'].mean()
df_2026_clean['USG_PCT'] = df_2026_clean['USG_PCT'].fillna(league_avg_usg_2026)

print(f"NaNs in USG_PCT: {df_2026_clean['USG_PCT'].isna().sum()}")
print(f"League avg USG (2026): {league_avg_usg_2026:.3f}")

# Team usage context
for col in ['TEAM_AVG_USG', 'RELATIVE_USG', 'USG_RANK']:
    if col in df_2026_clean.columns:
        df_2026_clean = df_2026_clean.drop(columns=[col])

team_avg_usg_2026 = df_2026_clean.groupby(
    ['PLAYER_TEAM', 'SEASON']
)['USG_PCT'].mean().reset_index().rename(columns={'USG_PCT': 'TEAM_AVG_USG'})

df_2026_clean = df_2026_clean.merge(
    team_avg_usg_2026, on=['PLAYER_TEAM', 'SEASON'], how='left'
)
df_2026_clean['RELATIVE_USG'] = (
    df_2026_clean['USG_PCT'] / df_2026_clean['TEAM_AVG_USG']
)

player_season_usg_2026 = df_2026_clean.groupby(
    ['PLAYER_NAME', 'PLAYER_TEAM', 'SEASON']
)['USG_PCT'].first().reset_index()
player_season_usg_2026['USG_RANK'] = player_season_usg_2026.groupby(
    ['PLAYER_TEAM', 'SEASON']
)['USG_PCT'].rank(ascending=False, method='min')

df_2026_clean = df_2026_clean.merge(
    player_season_usg_2026[['PLAYER_NAME', 'PLAYER_TEAM', 'SEASON', 'USG_RANK']],
    on=['PLAYER_NAME', 'PLAYER_TEAM', 'SEASON'],
    how='left'
)

# Volatility features
for col in ['PTS_std_roll10', 'REB_std_roll10', 'PTS_cv_roll10', 'REB_cv_roll10']:
    if col in df_2026_clean.columns:
        df_2026_clean = df_2026_clean.drop(columns=[col])

df_2026_clean = df_2026_clean.sort_values(
    ['PLAYER_NAME', 'GAME_DATE']
).reset_index(drop=True)

df_2026_clean['PTS_std_roll10'] = (
    df_2026_clean.groupby('PLAYER_NAME')['PTS']
    .transform(lambda x: x.rolling(window=10, min_periods=3).std().shift(1))
)
df_2026_clean['REB_std_roll10'] = (
    df_2026_clean.groupby('PLAYER_NAME')['REB']
    .transform(lambda x: x.rolling(window=10, min_periods=3).std().shift(1))
)
df_2026_clean['PTS_cv_roll10'] = (
    df_2026_clean['PTS_std_roll10'] /
    df_2026_clean['PTS_roll10'].replace(0, np.nan)
)
df_2026_clean['REB_cv_roll10'] = (
    df_2026_clean['REB_std_roll10'] /
    df_2026_clean['REB_roll10'].replace(0, np.nan)
)

for col in ['PTS_std_roll10', 'REB_std_roll10', 'PTS_cv_roll10', 'REB_cv_roll10']:
    df_2026_clean[col] = df_2026_clean[col].fillna(df_2026_clean[col].mean())

if 'USG_PCT_roll5' in df_2026_clean.columns:
    df_2026_clean = df_2026_clean.drop(columns=['USG_PCT_roll5'])

df_2026_clean['USG_PCT_roll5'] = (
    df_2026_clean.groupby('PLAYER_NAME')['USG_PCT']
    .transform(lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))
)
df_2026_clean['USG_PCT_roll5'] = df_2026_clean['USG_PCT_roll5'].fillna(
    df_2026_clean['USG_PCT']
)

# Normalize rolling pos features
for col in ['OPP_PTS_VS_POS_roll5', 'OPP_REB_VS_POS_roll5',
            'OPP_AST_VS_POS_roll5', 'OPP_BLK_VS_POS_roll5',
            'OPP_STL_VS_POS_roll5', 'OPP_3PM_VS_POS_roll5']:
    norm_col = f'{col}_norm'
    if norm_col in df_2026_clean.columns:
        df_2026_clean = df_2026_clean.drop(columns=[norm_col])
    df_2026_clean[norm_col] = df_2026_clean.groupby('POSITION')[col].transform(
        lambda x: (x - x.mean()) / (x.std() + 1e-8)
    )
    df_2026_clean[norm_col] = df_2026_clean[norm_col].fillna(0)

print()
print(f"Total columns: {len(df_2026_clean.columns)}")
print()

check_cols = [
    'OPP_PTS_VS_POS', 'OPP_PTS_VS_POS_roll5_norm', 'USG_PCT_roll5',
    'RELATIVE_USG', 'USG_RANK', 'PTS_std_roll10', 'PTS_cv_roll10',
    'IS_ROOKIE_SEASON', 'GAMES_PLAYED'
]

print("Feature check:")
for col in check_cols:
    nans   = df_2026_clean[col].isna().sum()
    status = "✅" if nans == 0 else f"❌ {nans} NaNs"
    print(f"  {col:<35} {status}")

NaNs in USG_PCT: 0
League avg USG (2026): 0.180

Total columns: 106

Feature check:
  OPP_PTS_VS_POS                      ✅
  OPP_PTS_VS_POS_roll5_norm           ✅
  USG_PCT_roll5                       ✅
  RELATIVE_USG                        ✅
  USG_RANK                            ✅
  PTS_std_roll10                      ✅
  PTS_cv_roll10                       ✅
  IS_ROOKIE_SEASON                    ✅
  GAMES_PLAYED                        ✅


In [31]:
# Check which columns exist in df_2026_clean but not in df_master_wnba
extra_cols = [col for col in df_2026_clean.columns if col not in df_master_wnba.columns]
missing_cols = [col for col in df_master_wnba.columns if col not in df_2026_clean.columns]

print(f"Columns in df_2026_clean but NOT in df_master_wnba ({len(extra_cols)}):")
for col in extra_cols:
    print(f"  {col}")
print()
print(f"Columns in df_master_wnba but NOT in df_2026_clean ({len(missing_cols)}):")
for col in missing_cols:
    print(f"  {col}")

Columns in df_2026_clean but NOT in df_master_wnba (14):
  OPP_PTS_VS_POS_x
  OPP_REB_VS_POS_x
  OPP_AST_VS_POS_x
  OPP_BLK_VS_POS_x
  OPP_STL_VS_POS_x
  OPP_3PM_VS_POS_x
  USG_PCT_x
  OPP_PTS_VS_POS_y
  OPP_REB_VS_POS_y
  OPP_AST_VS_POS_y
  OPP_BLK_VS_POS_y
  OPP_STL_VS_POS_y
  OPP_3PM_VS_POS_y
  USG_PCT_y

Columns in df_master_wnba but NOT in df_2026_clean (0):


In [33]:
# Drop stale duplicate columns from earlier failed merge attempts
extra_cols = [col for col in df_2026_clean.columns if col not in df_master_wnba.columns]
df_2026_clean = df_2026_clean.drop(columns=extra_cols)

print(f"Dropped {len(extra_cols)} stale columns")
print(f"Remaining columns: {len(df_2026_clean.columns)}")
print()

# Final alignment check
extra_check   = [col for col in df_2026_clean.columns if col not in df_master_wnba.columns]
missing_check = [col for col in df_master_wnba.columns if col not in df_2026_clean.columns]

print(f"Extra columns remaining:   {extra_check if extra_check else 'None ✅'}")
print(f"Missing columns remaining: {missing_check if missing_check else 'None ✅'}")

Dropped 14 stale columns
Remaining columns: 92

Extra columns remaining:   None ✅
Missing columns remaining: None ✅


In [35]:
# Append 2026 data to master dataset and save

print(f"Existing rows: {len(df_master_wnba):,}")
print(f"2026 rows to add: {len(df_2026_clean):,}")
print()

# Align columns — keep only columns that exist in both
common_cols = [col for col in df_master_wnba.columns if col in df_2026_clean.columns]
missing_in_2026 = [col for col in df_master_wnba.columns if col not in df_2026_clean.columns]

if len(missing_in_2026) > 0:
    print(f"Columns in master but not in 2026 data:")
    for col in missing_in_2026:
        print(f"  {col}")
    print()

# Append
df_master_wnba = pd.concat(
    [df_master_wnba, df_2026_clean[common_cols]],
    ignore_index=True
)

# Sort by player and date
df_master_wnba = df_master_wnba.sort_values(
    ['PLAYER_NAME', 'GAME_DATE']
).reset_index(drop=True)

print(f"Total rows after append: {len(df_master_wnba):,}")
print()
print("Rows per season:")
print(df_master_wnba.groupby('SEASON').size())
print()

Existing rows: 14,597
2026 rows to add: 1,285

Total rows after append: 15,882

Rows per season:
SEASON
2021    2335
2022    2706
2023    2893
2024    3021
2025    3642
2026    1285
dtype: int64



In [37]:
# Add head-to-head history features across full combined dataset
df_master_wnba = df_master_wnba.sort_values(['PLAYER_NAME', 'GAME_DATE']).reset_index(drop=True)

h2h_stats = ['PTS', 'REB', 'AST', 'FG3M']

for stat in h2h_stats:
    col = f'H2H_{stat}_AVG'
    if col in df_master_wnba.columns:
        df_master_wnba = df_master_wnba.drop(columns=[col])

    df_master_wnba[col] = df_master_wnba.groupby(
        ['PLAYER_NAME', 'OPPONENT']
    )[stat].transform(
        lambda x: x.shift(1).rolling(window=5, min_periods=1).mean()
    )

# Number of previous games vs this opponent (before this game)
if 'H2H_GAMES' in df_master_wnba.columns:
    df_master_wnba = df_master_wnba.drop(columns=['H2H_GAMES'])

df_master_wnba['H2H_GAMES'] = df_master_wnba.groupby(
    ['PLAYER_NAME', 'OPPONENT']
).cumcount()

# Fill NaNs for first-time matchups using player's overall rolling average
for stat in h2h_stats:
    col = f'H2H_{stat}_AVG'
    df_master_wnba[col] = df_master_wnba[col].fillna(
        df_master_wnba[f'{stat}_roll5']
    )

print(f"Total columns now: {len(df_master_wnba.columns)}")
print()

# Check NaNs
h2h_cols = [f'H2H_{s}_AVG' for s in h2h_stats] + ['H2H_GAMES']
print("H2H feature check:")
for col in h2h_cols:
    nans   = df_master_wnba[col].isna().sum()
    status = "✅" if nans == 0 else f"❌ {nans} NaNs"
    print(f"  {col:<20} {status}")

print()

# Spot check — A'ja Wilson vs a frequent opponent
sample = df_master_wnba[
    df_master_wnba['PLAYER_NAME'] == "A'ja Wilson"
][['GAME_DATE', 'OPPONENT', 'PTS', 'H2H_PTS_AVG', 'H2H_GAMES']].tail(15)
print("A'ja Wilson — recent games with H2H features:")
print(sample.to_string(index=False))

Total columns now: 97

H2H feature check:
  H2H_PTS_AVG          ✅
  H2H_REB_AVG          ✅
  H2H_AST_AVG          ✅
  H2H_FG3M_AVG         ✅
  H2H_GAMES            ✅

A'ja Wilson — recent games with H2H features:
 GAME_DATE OPPONENT  PTS  H2H_PTS_AVG  H2H_GAMES
2025-09-11      LAS   23         30.2         14
2026-05-15      CON   45         24.4         15
2026-05-17      ATL   20         29.8         15
2026-05-23      LAS   24         27.8         15
2026-05-28      DAL   21         31.0         16
2026-05-31      GSV   28         23.0          4
2026-06-02      LAS   25         25.8         16
2026-06-06      GSV   28         24.0          5
2026-06-08      SEA   34         20.8         15
2026-06-11      PDX   32         27.2          0
2026-06-13      MIN   24         19.0         18
2026-06-15      DAL   18         29.6         17
2026-06-17      PHX   33         25.0          3
2026-06-21      GSV   19         26.2          6
2026-06-23      NYL   16         17.4         14


In [39]:
# Save updated dataset with H2H features
df_master_wnba.to_csv('wnba_master_dataset.csv', index=False)

df_verify = pd.read_csv('wnba_master_dataset.csv', nrows=1)

print(f"✅ Saved successfully")
print(f"Rows:    {len(df_master_wnba):,}")
print(f"Columns: {len(df_verify.columns)}")
print()

print("Final summary:")
print(f"  Total players: {df_master_wnba['PLAYER_NAME'].nunique()}")
print(f"  Seasons:       {sorted(df_master_wnba['SEASON'].unique())}")
print(f"  2026 rookies:  {df_master_wnba[(df_master_wnba['SEASON']==2026) & (df_master_wnba['IS_ROOKIE_SEASON']==1)]['PLAYER_NAME'].nunique()}")

✅ Saved successfully
Rows:    15,882
Columns: 97

Final summary:
  Total players: 204
  Seasons:       [2021, 2022, 2023, 2024, 2025, 2026]
  2026 rookies:  24


In [1]:
# Minimal setup — load saved dataset instead of rebuilding from scratch
import pandas as pd
import time

df_master_wnba = pd.read_csv('wnba_master_dataset.csv', parse_dates=['GAME_DATE'])
print(f"✅ Loaded {len(df_master_wnba):,} rows")

team_name_map = {
    'ATL': 'Atlanta Dream', 'CHI': 'Chicago Sky', 'CON': 'Connecticut Sun',
    'DAL': 'Dallas Wings', 'GSV': 'Golden State Valkyries',
    'IND': 'Indiana Fever', 'LAS': 'Los Angeles Sparks', 'LVA': 'Las Vegas Aces',
    'MIN': 'Minnesota Lynx', 'NYL': 'New York Liberty',
    'PHO': 'Phoenix Mercury', 'PHX': 'Phoenix Mercury',
    'SEA': 'Seattle Storm', 'WAS': 'Washington Mystics',
    'POR': 'Portland Fire', 'TOR': 'Toronto Tempo',
}

def clean_player_log_wnba(df, player_name, season, first_season=False):
    df['PLAYER_NAME'] = player_name
    df['SEASON']      = season
    df['GAME_DATE']   = pd.to_datetime(df['GAME_DATE'], format='mixed')
    df['HOME_AWAY']   = df['MATCHUP'].apply(lambda x: 'HOME' if 'vs.' in x else 'AWAY')
    df = df[df['MIN'] >= 12].copy()
    df = df.sort_values('GAME_DATE').reset_index(drop=True)
    df['GAMES_PLAYED'] = range(1, len(df) + 1)
    df['IS_ROOKIE_SEASON'] = 1 if first_season else 0
    target_stats       = ['PTS', 'REB', 'AST', 'BLK', 'STL', 'FG3M']
    feature_only_stats = ['MIN', 'TOV', 'FGA', 'FG3A']
    for stat in target_stats + feature_only_stats:
        df[f'{stat}_roll5']  = df[stat].rolling(window=5, min_periods=3).mean().shift(1)
        df[f'{stat}_roll10'] = df[stat].rolling(window=10, min_periods=5).mean().shift(1)
    df = df.dropna(subset=['PTS_roll5']).reset_index(drop=True)
    return df

print("✅ Setup complete")

✅ Loaded 15,200 rows
✅ Setup complete


In [7]:
# Remove old partial 2026 data — fixed dtype comparison
rows_before = len(df_master_wnba)
df_master_wnba = df_master_wnba[df_master_wnba['SEASON'] != 2026].copy()
rows_removed = rows_before - len(df_master_wnba)

print(f"Removed {rows_removed} old 2026 rows")
print(f"Remaining rows (2021-2025): {len(df_master_wnba):,}")

Removed 603 old 2026 rows
Remaining rows (2021-2025): 14,597


In [9]:
# Step 2 — Pull current 2026 team defensive and pace stats
# Now meaningful with 16+ games per team — no longer need 2025 proxy
from nba_api.stats.endpoints import leaguedashteamstats

stats_def_2026 = leaguedashteamstats.LeagueDashTeamStats(
    season='2026',
    season_type_all_star='Regular Season',
    measure_type_detailed_defense='Defense',
    league_id_nullable='10'
)
time.sleep(1.5)
df_def_2026 = stats_def_2026.get_data_frames()[0]
df_def_2026['SEASON'] = 2026
df_def_2026 = df_def_2026[['TEAM_NAME', 'SEASON', 'DEF_RATING']].copy()

stats_pace_2026 = leaguedashteamstats.LeagueDashTeamStats(
    season='2026',
    season_type_all_star='Regular Season',
    measure_type_detailed_defense='Advanced',
    league_id_nullable='10'
)
time.sleep(1.5)
df_pace_2026 = stats_pace_2026.get_data_frames()[0]
df_pace_2026['SEASON'] = 2026
df_pace_2026 = df_pace_2026[['TEAM_NAME', 'SEASON', 'PACE']].copy()

stats_base_2026 = leaguedashteamstats.LeagueDashTeamStats(
    season='2026',
    season_type_all_star='Regular Season',
    measure_type_detailed_defense='Base',
    per_mode_detailed='PerGame',
    league_id_nullable='10'
)
time.sleep(1.5)
df_base_2026 = stats_base_2026.get_data_frames()[0]
df_base_2026['SEASON'] = 2026
df_opp_stats_2026 = df_base_2026[[
    'TEAM_NAME', 'SEASON', 'PTS', 'REB', 'AST', 'BLK', 'STL', 'FG3M'
]].rename(columns={
    'PTS': 'OPP_PTS_ALLOWED_PG', 'REB': 'OPP_REB_ALLOWED_PG',
    'AST': 'OPP_AST_ALLOWED_PG', 'BLK': 'OPP_BLK_PG',
    'STL': 'OPP_STL_PG', 'FG3M': 'OPP_3PM_ALLOWED_PG',
})

print(f"2026 teams with stats: {len(df_def_2026)}")
print()
df_team_check = df_def_2026.merge(df_pace_2026, on=['TEAM_NAME', 'SEASON'])
df_team_check = df_team_check.merge(df_opp_stats_2026, on=['TEAM_NAME', 'SEASON'])
print(df_team_check.to_string())

2026 teams with stats: 15

                 TEAM_NAME  SEASON  DEF_RATING   PACE  OPP_PTS_ALLOWED_PG  OPP_REB_ALLOWED_PG  OPP_AST_ALLOWED_PG  OPP_BLK_PG  OPP_STL_PG  OPP_3PM_ALLOWED_PG
0            Atlanta Dream    2026       103.7  96.45                90.4                35.7                20.4         2.9         9.4                 9.2
1              Chicago Sky    2026       107.1  98.72                82.4                33.1                18.9         5.5         6.9                 6.4
2          Connecticut Sun    2026       109.2  96.40                79.9                34.1                18.8         4.4         8.1                 5.1
3             Dallas Wings    2026       105.1  96.07                89.4                34.0                23.1         3.4         7.6                 8.5
4   Golden State Valkyries    2026       103.6  92.15                83.8                33.9                18.4         4.0         6.9                11.1
5            Indiana Feve